# Phân tích trường `Source` bị thiếu — ngày 16

Notebook kiểm tra Event 4624 và 4625, tập trung vào giả thuyết `Source` bị thiếu có liên hệ với `LogonType` và `AuthenticationPackage`. Dữ liệu được đọc theo batch để không phải nạp toàn bộ file vào RAM.

Các chỉ số chính:

- `MissingRatePct`: xác suất `Source` bị thiếu bên trong một nhóm.
- `ShareOfAllMissingPct`: tỷ trọng của nhóm trong toàn bộ các dòng thiếu `Source`.
- `LiftVsOverall`: tỷ lệ thiếu của nhóm chia cho tỷ lệ thiếu chung; lớn hơn 1 nghĩa là thiếu nhiều hơn mức nền.
- `CramersV`: cường độ liên hệ giữa biến phân loại và trạng thái thiếu `Source`; gần 0 là yếu, càng gần 1 càng mạnh.

In [1]:
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

PARQUET_RELATIVE_PATHS = {
    "Event 4624": Path("data/interim/event_4624/event_4624_day-16.parquet"),
    "Event 4625": Path("data/interim/event_4625/event_4625_day-16.parquet"),
}
TARGET_COLUMN = "Source"
BATCH_SIZE = 250_000
MISSING_LABEL = "<MISSING>"
MAX_JOINT_ROWS_TO_DISPLAY = 30

GROUPS = {
    "LogonType": ("LogonType", "LogonTypeDescription"),
    "AuthenticationPackage": ("AuthenticationPackage",),
    "LogonType + AuthenticationPackage": (
        "LogonType",
        "LogonTypeDescription",
        "AuthenticationPackage",
    ),
}


def find_project_file(relative_path: Path, start_directory: Optional[Path] = None) -> Path:
    """Tìm file từ thư mục hiện tại hoặc một trong các thư mục cha."""
    start_directory = (start_directory or Path.cwd()).resolve()
    for directory in (start_directory, *start_directory.parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Không tìm thấy '{relative_path}' từ '{start_directory}' hoặc các thư mục cha."
    )


parquet_paths = {
    event_name: find_project_file(relative_path)
    for event_name, relative_path in PARQUET_RELATIVE_PATHS.items()
}
parquet_paths

{'Event 4624': WindowsPath('D:/Github Repo/ueba-benchmark/data/interim/event_4624/event_4624_day-16.parquet'),
 'Event 4625': WindowsPath('D:/Github Repo/ueba-benchmark/data/interim/event_4625/event_4625_day-16.parquet')}

In [2]:
def missing_text_mask(series: pd.Series) -> pd.Series:
    """Nhận diện null, chuỗi rỗng và chuỗi chỉ chứa khoảng trắng."""
    text = series.astype("string")
    return series.isna() | text.str.strip().eq("").fillna(False)


def normalize_group_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa các chiều phân tích để groupby không làm mất nhóm null."""
    normalized = frame.copy()
    normalized["LogonType"] = (
        normalized["LogonType"].astype("Int64").astype("string").fillna(MISSING_LABEL)
    )
    for column in ("LogonTypeDescription", "AuthenticationPackage"):
        normalized[column] = normalized[column].astype("string").fillna(MISSING_LABEL)
    return normalized


def update_counter(
    counter: Counter,
    frame: pd.DataFrame,
    dimensions: tuple[str, ...],
) -> None:
    """Cộng số lượng của một batch vào Counter tổng."""
    counts = frame.groupby(list(dimensions), observed=True, dropna=False).size()
    counter.update({key: int(value) for key, value in counts.items()})


def analyze_source_missingness(
    file_path: Path,
    batch_size: int = 250_000,
) -> dict:
    """Tính thống kê thiếu Source theo các chiều mà không giữ toàn bộ dữ liệu trong RAM."""
    required_columns = [
        TARGET_COLUMN,
        "LogonType",
        "LogonTypeDescription",
        "AuthenticationPackage",
    ]
    parquet_file = pq.ParquetFile(file_path)
    missing_columns = set(required_columns) - set(parquet_file.schema_arrow.names)
    if missing_columns:
        raise KeyError(f"File thiếu các cột bắt buộc: {sorted(missing_columns)}")

    total_counts = {name: Counter() for name in GROUPS}
    missing_counts = {name: Counter() for name in GROUPS}
    total_rows = 0
    missing_rows = 0

    for batch in parquet_file.iter_batches(
        columns=required_columns,
        batch_size=batch_size,
    ):
        frame = batch.to_pandas()
        source_is_missing = missing_text_mask(frame[TARGET_COLUMN])
        normalized = normalize_group_columns(frame)

        total_rows += len(normalized)
        missing_rows += int(source_is_missing.sum())

        for group_name, dimensions in GROUPS.items():
            update_counter(total_counts[group_name], normalized, dimensions)
            update_counter(
                missing_counts[group_name],
                normalized.loc[source_is_missing],
                dimensions,
            )

    return {
        "file_path": file_path,
        "total_rows": total_rows,
        "missing_rows": missing_rows,
        "total_counts": total_counts,
        "missing_counts": missing_counts,
    }


def build_group_table(result: dict, group_name: str) -> pd.DataFrame:
    """Tạo bảng tỷ lệ thiếu, tỷ trọng và lift cho một chiều phân tích."""
    dimensions = GROUPS[group_name]
    overall_rate = result["missing_rows"] / result["total_rows"]
    rows = []

    for key, total_count in result["total_counts"][group_name].items():
        key_values = key if isinstance(key, tuple) else (key,)
        missing_count = result["missing_counts"][group_name].get(key, 0)
        missing_rate = missing_count / total_count
        row = dict(zip(dimensions, key_values))
        row.update(
            TotalCount=int(total_count),
            MissingCount=int(missing_count),
            PresentCount=int(total_count - missing_count),
            MissingRatePct=100 * missing_rate,
            ShareOfAllMissingPct=(
                100 * missing_count / result["missing_rows"]
                if result["missing_rows"]
                else 0.0
            ),
            LiftVsOverall=(missing_rate / overall_rate if overall_rate else np.nan),
        )
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(["MissingCount", "MissingRatePct"], ascending=False)
        .reset_index(drop=True)
    )

In [3]:
analysis_by_event = {
    event_name: analyze_source_missingness(parquet_path, batch_size=BATCH_SIZE)
    for event_name, parquet_path in parquet_paths.items()
}

overview = pd.DataFrame(
    [
        {
            "Event": event_name,
            "TotalRows": result["total_rows"],
            "MissingSource": result["missing_rows"],
            "PresentSource": result["total_rows"] - result["missing_rows"],
            "MissingRatePct": 100 * result["missing_rows"] / result["total_rows"],
        }
        for event_name, result in analysis_by_event.items()
    ]
)
display(overview.round({"MissingRatePct": 3}))

,Event,TotalRows,MissingSource,PresentSource,MissingRatePct
0,Event 4624,17553517,4261773,13291744,24.279
1,Event 4625,359897,115473,244424,32.085


In [4]:
ROUNDING = {
    "MissingRatePct": 3,
    "ShareOfAllMissingPct": 3,
    "LiftVsOverall": 2,
}

group_tables = {}
for event_name, result in analysis_by_event.items():
    print("\n" + "=" * 100)
    print(event_name)
    print("=" * 100)
    group_tables[event_name] = {}

    for group_name in GROUPS:
        table = build_group_table(result, group_name)
        group_tables[event_name][group_name] = table
        print(f"\nPhân tích theo {group_name}")
        rows_to_show = (
            MAX_JOINT_ROWS_TO_DISPLAY
            if group_name == "LogonType + AuthenticationPackage"
            else len(table)
        )
        display(table.head(rows_to_show).round(ROUNDING))


Event 4624

Phân tích theo LogonType


,LogonType,LogonTypeDescription,TotalCount,MissingCount,PresentCount,MissingRatePct,ShareOfAllMissingPct,LiftVsOverall
0,3,Network,16794417,3866110,12928307,23.020,90.716,0.95
1,5,Service,373734,373449,285,99.924,8.763,4.12
2,9,NewCredentials,13451,13451,0,100.000,0.316,4.12
3,0,Local System,5786,5786,0,100.000,0.136,4.12
4,2,Interactive,219613,2178,217435,0.992,0.051,0.04
5,10,RemoteInteractive,2230,792,1438,35.516,0.019,1.46
6,4,Batch,2321,4,2317,0.172,0.000,0.01
7,5,<MISSING>,3,3,0,100.000,0.000,4.12
8,7,Unlock,78760,0,78760,0.000,0.000,0.00
9,8,NetworkClearText,62286,0,62286,0.000,0.000,0.00



Phân tích theo AuthenticationPackage


,AuthenticationPackage,TotalCount,MissingCount,PresentCount,MissingRatePct,ShareOfAllMissingPct,LiftVsOverall
0,Kerberos,13483340,3859981,9623359,28.628,90.572,1.18
1,Negotiate,958030,389877,568153,40.696,9.148,1.68
2,NTLM,3046192,5999,3040193,0.197,0.141,0.01
3,-,5786,5786,0,100.000,0.136,4.12
4,Microsoft Unified Security Protocol Provider,128,128,0,100.000,0.003,4.12
5,MICROSOFT_AUTHENTICATION_PACKAGE_V1_0,60041,2,60039,0.003,0.000,0.00



Phân tích theo LogonType + AuthenticationPackage


...

,LogonType,LogonTypeDescription,TotalCount,MissingCount,PresentCount,MissingRatePct,ShareOfAllMissingPct,LiftVsOverall
0,3,Network,334377,115155,219222,34.439,99.725,1.07
1,10,RemoteInteractive,267,267,0,100.000,0.231,3.12
2,5,Service,146,46,100,31.507,0.040,0.98
3,4,Batch,1906,3,1903,0.157,0.003,0.00
4,11,CachedInteractive,588,1,587,0.170,0.001,0.01
5,2,Interactive,18872,1,18871,0.005,0.001,0.00
6,7,Unlock,3624,0,3624,0.000,0.000,0.00
7,8,NetworkClearText,117,0,117,0.000,0.000,0.00



Phân tích theo AuthenticationPackage


,AuthenticationPackage,TotalCount,MissingCount,PresentCount,MissingRatePct,ShareOfAllMissingPct,LiftVsOverall
0,NTLM,173382,111613,61769,64.374,96.657,2.01
1,Kerberos,160579,3542,157037,2.206,3.067,0.07
2,Negotiate,25799,318,25481,1.233,0.275,0.04
3,MICROSOFT_AUTHENTICATION_PACKAGE_V1_0,137,0,137,0.000,0.000,0.00



Phân tích theo LogonType + AuthenticationPackage


,LogonType,LogonTypeDescription,AuthenticationPackage,TotalCount,MissingCount,PresentCount,MissingRatePct,ShareOfAllMissingPct,LiftVsOverall
0,3,Network,NTLM,173382,111613,61769,64.374,96.657,2.01
1,3,Network,Kerberos,160579,3542,157037,2.206,3.067,0.07
2,10,RemoteInteractive,Negotiate,267,267,0,100.000,0.231,3.12
3,5,Service,Negotiate,146,46,100,31.507,0.040,0.98
4,4,Batch,Negotiate,1905,3,1902,0.157,0.003,0.00
5,11,CachedInteractive,Negotiate,588,1,587,0.170,0.001,0.01
6,2,Interactive,Negotiate,18859,1,18858,0.005,0.001,0.00
7,2,Interactive,MICROSOFT_AUTHENTICATION_PACKAGE_V1_0,13,0,13,0.000,0.000,0.00
8,3,Network,MICROSOFT_AUTHENTICATION_PACKAGE_V1_0,116,0,116,0.000,0.000,0.00
9,3,Network,Negotiate,300,0,300,0.000,0.000,0.00


In [5]:
def cramers_v_for_missingness(table: pd.DataFrame) -> float:
    """Tính Cramér's V giữa nhóm phân loại và trạng thái Source thiếu/có."""
    observed = table[["MissingCount", "PresentCount"]].to_numpy(dtype=float)
    observed = observed[observed.sum(axis=1) > 0]
    if observed.shape[0] < 2 or observed.sum() == 0:
        return 0.0

    expected = (
        observed.sum(axis=1, keepdims=True)
        @ observed.sum(axis=0, keepdims=True)
        / observed.sum()
    )
    valid = expected > 0
    chi_squared = np.sum(((observed - expected) ** 2)[valid] / expected[valid])
    denominator = observed.sum() * min(observed.shape[0] - 1, observed.shape[1] - 1)
    return float(np.sqrt(chi_squared / denominator)) if denominator else 0.0


def describe_strength(value: float) -> str:
    """Nhãn diễn giải thực dụng cho effect size; không phải ranh giới tuyệt đối."""
    if value < 0.10:
        return "Yếu"
    if value < 0.30:
        return "Vừa"
    return "Mạnh"


association_rows = []
for event_name, tables in group_tables.items():
    for group_name, table in tables.items():
        value = cramers_v_for_missingness(table)
        association_rows.append(
            {
                "Event": event_name,
                "Factor": group_name,
                "CramersV": value,
                "Strength": describe_strength(value),
            }
        )

association_table = pd.DataFrame(association_rows)
display(association_table.round({"CramersV": 4}))

,Event,Factor,CramersV,Strength
0,Event 4624,LogonType,0.2772,Vừa
1,Event 4624,AuthenticationPackage,0.2698,Vừa
2,Event 4624,LogonType + AuthenticationPackage,0.3788,Mạnh
3,Event 4625,LogonType,0.1920,Vừa
4,Event 4625,AuthenticationPackage,0.6669,Mạnh
5,Event 4625,LogonType + AuthenticationPackage,0.6696,Mạnh


## Cách kết luận pattern

1. Xem `CramersV` để đánh giá mức liên hệ tổng thể. Với dữ liệu rất lớn, effect size hữu ích hơn việc chỉ dựa vào kiểm định có ý nghĩa thống kê.
2. Trong bảng tổ hợp, ưu tiên nhóm vừa có `MissingRatePct` cao, vừa có `MissingCount` đủ lớn.
3. `LiftVsOverall > 1` cho biết tổ hợp thiếu `Source` nhiều hơn mức nền của chính loại event đó.
4. Không kết luận nguyên nhân chỉ từ tương quan. `LogonType` và `AuthenticationPackage` có thể là dấu hiệu đại diện cho một luồng xác thực cụ thể chứ không trực tiếp làm mất `Source`.

In [6]:
def arrow_missing_text_mask(values: pa.Array) -> pa.Array:
    """Mask Arrow cho null, chuỗi rỗng và chuỗi chỉ chứa khoảng trắng."""
    values_without_null = pc.fill_null(values, "")
    return pc.equal(pc.utf8_trim_whitespace(values_without_null), "")


def read_missing_source_examples(
    file_path: Path,
    limit: int = 10,
    batch_size: int = 250_000,
) -> pd.DataFrame:
    """Lấy một số dòng thiếu Source và dừng ngay khi đủ mẫu."""
    parquet_file = pq.ParquetFile(file_path)
    source_index = parquet_file.schema_arrow.names.index(TARGET_COLUMN)
    frames = []
    remaining = limit

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        mask = arrow_missing_text_mask(batch.column(source_index))
        if pc.any(mask).as_py():
            matches = pa.Table.from_batches([batch]).filter(mask).slice(0, remaining)
            frames.append(matches.to_pandas())
            remaining -= matches.num_rows
        if remaining == 0:
            break

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


for event_name, parquet_path in parquet_paths.items():
    print(f"\nVí dụ {event_name} thiếu Source")
    display(read_missing_source_examples(parquet_path, limit=10, batch_size=BATCH_SIZE))


Ví dụ Event 4624 thiếu Source


,Time,EventID,LogHost,LogonType,LogonTypeDescription,UserName,DomainName,LogonID,SubjectUserName,SubjectDomainName,...,Status,Source,ServiceName,Destination,AuthenticationPackage,FailureReason,ProcessName,ProcessID,ParentProcessName,ParentProcessID
0,1296000,4624,Comp284280,3,Network,Comp284280$,Domain001,0x9ea1df6,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN
1,1296000,4624,Comp505809,3,Network,Comp505809$,Domain001,0x1b9dba9f,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN
2,1296000,4624,Comp995416,5,Service,system,nt authority,0x3e7,NaN,NaN,...,NaN,NaN,NaN,NaN,Negotiate,NaN,services.exe,0x2bc,NaN,NaN
3,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN
4,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN
5,1296000,4624,Comp186428,3,Network,User643724,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN
6,1296000,4624,Comp861045,5,Service,system,nt authority,0x3e7,NaN,NaN,...,NaN,NaN,NaN,NaN,Negotiate,NaN,services.exe,0x2bc,NaN,NaN
7,1296000,4624,Comp635310,5,Service,system,nt authority,0x3e7,NaN,NaN,...,NaN,NaN,NaN,NaN,Negotiate,NaN,services.exe,0x278,NaN,NaN
8,1296000,4624,Comp798573,5,Service,system,nt authority,0x3e7,NaN,NaN,...,NaN,NaN,NaN,NaN,Negotiate,NaN,services.exe,0x2d8,NaN,NaN
9,1296000,4624,Comp208293,3,Network,Comp208293$,Domain001,0x218fa896,NaN,NaN,...,NaN,NaN,NaN,NaN,Kerberos,NaN,NaN,NaN,NaN,NaN



Ví dụ Event 4625 thiếu Source


,Time,EventID,LogHost,LogonType,LogonTypeDescription,UserName,DomainName,LogonID,SubjectUserName,SubjectDomainName,...,Status,Source,ServiceName,Destination,AuthenticationPackage,FailureReason,ProcessName,ProcessID,ParentProcessName,ParentProcessID
0,1296015,4625,Comp180216,3,Network,Scanner,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,The user has not been granted the requested lo...,NaN,NaN,NaN,NaN
1,1296015,4625,Comp180216,3,Network,Scanner,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,The user has not been granted the requested lo...,NaN,NaN,NaN,NaN
2,1296015,4625,Comp180216,3,Network,Scanner,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,The user has not been granted the requested lo...,NaN,NaN,NaN,NaN
3,1296015,4625,Comp180216,3,Network,Scanner,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,The user has not been granted the requested lo...,NaN,NaN,NaN,NaN
4,1296019,4625,Comp984447,3,Network,Administrator,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
5,1296019,4625,Comp148386,3,Network,Administrator,Domain001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
6,1296020,4625,Comp064174,3,Network,Administrator,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
7,1296020,4625,Comp397519,3,Network,Administrator,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
8,1296021,4625,Comp108190,3,Network,Administrator,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
9,1296022,4625,Comp989334,3,Network,Administrator,Domain002,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NTLM,Unknown user name or bad password.,NaN,NaN,NaN,NaN
